# Formation Heat Transfer — Coupled Completion/Formation Model (Phase 4)

N₂ blowdown of the LSU CO₂ research well (7-5/8" CRA inner casing,
ID 170.3 mm, 2,000 ft) with the conjugate radial thermal model: casing →
air annulus (Willhite natural convection + radiation) → 13-3/8" casing →
cement → 20" casing → cement → formation (geothermal far field).
Formulation and verification: `docs/thermal_model.md`.

In [ ]:
try:
    import hyperbolic1d
except ImportError:
    %pip install -q "hyperbolic1d[plot] @ git+https://github.com/JaderBarbosaLSU/hyperco2-wellbore"
import numpy as np, matplotlib.pyplot as plt
from hyperbolic1d.wellbore import (WellboreModel, WellGeometry, ChokeValve,
                                   AdiabaticWall, PrescribedWallTemperature)
from hyperbolic1d.thermal import (FormationCoupledWall, lsu_co2_well_stack,
                                  GEOTHERMAL_DEFAULT)

## 1. LSU well stack and model setup

In [ ]:
r1, layers, formation, r_far = lsu_co2_well_stack()
for l in layers:
    print(f"{getattr(l,'name','?'):22s} {l.r_in*1e3:7.1f} -> {l.r_out*1e3:7.1f} mm")
geom  = WellGeometry(depth=609.6, diameter=2*r1)
choke = ChokeValve(diameter=0.0127, cd=0.84)
P0, T0, T_END = 1.379e7, 300.0, 1800.0
N = 200          # 300 used for the archived results; 200 is faster

## 2. Thermal hierarchy: adiabatic vs prescribed-T vs coupled

In [ ]:
zc = (np.arange(N) + 0.5) * geom.depth / N
T_geo = GEOTHERMAL_DEFAULT.T(geom.depth - zc)
cases = {"adiabatic": AdiabaticWall(),
         "prescribed $T_w$": PrescribedWallTemperature(T_wall=T_geo),
         "coupled formation": FormationCoupledWall()}
res = {}
for name, wall in cases.items():
    m = WellboreModel(geometry=geom, choke=choke, gravity=True,
                      n_cells=N, wall_model=wall)
    res[name] = m.solve(P0, T0, T_END, n_snapshots=5)
    r = res[name]
    print(f"{name:20s} T_wh min={r.T_wellhead.min():6.1f} K   "
          f"budgets m={r.mass_balance_error:.1e} E={r.energy_balance_error:.1e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))
for name, r in res.items():
    axes[0].plot(r.t_hist, r.T_wellhead, label=name)
    axes[1].plot(r.t_hist, r.mdot)
    axes[2].plot(r.t_hist, r.mass)
for ax, lab in zip(axes, ["Wellhead T [K]", "$\\dot m$ [kg/s]", "Mass in well [kg]"]):
    ax.set_xlabel("t [s]"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8); fig.tight_layout()

Heat ingress from the completion/formation nearly suppresses the
expansion cooling for this slow (1/2-in choke) blowdown. The result is
choke-size dependent: larger bores shift the behavior toward adiabatic.

## 3. Radial temperature field (coupled model)

In [ ]:
wall = cases["coupled formation"]
Tr, rr = wall.radial_temperatures, wall.radial_positions
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for frac, lab in ((0.95, "near wellhead"), (0.5, "mid depth"), (0.05, "near bottom")):
    j = int(frac*(N-1))
    ax.semilogx(rr, Tr[j], ".-", ms=3, label=f"{lab} (z = {zc[j]:.0f} m)")
ax.set_xlabel("r [m]"); ax.set_ylabel("T [K]"); ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8); ax.set_title("Through-completion radial temperature at t = t_end")
# Most of the solid-side temperature drop occurs across the air annulus
# (96.8 -> 156.8 mm): it is the dominant resistance after the fluid film.

## 4. Experiments

* `ChokeValve(diameter=0.0508)` — fast blowdown: adiabatic-like behavior returns.
* `GasGap(..., eps_in=0.1, eps_out=0.1)` in a custom stack — low-emissivity
  surfaces: quantify the radiation share of the annulus conductance.
* `FormationCoupledWall(n_formation=32)` — radial resolution check.
* Replace the air gap by a `SolidLayer` of cement to mimic a cemented
  7-5/8 annulus and compare.